
# 🧬 Molecular Embedding Generation for ChemTastesDB

This notebook generates multiple types of molecular embeddings for the multi-label taste prediction task.

## Input
- `chemtastesdb_multilabel_clean.csv` - Cleaned dataset with SMILES and multi-label taste annotations

## Output Files (Separate for each embedding type)
| File | Embedding Type | Dimensions | Description |
|------|---------------|------------|-------------|
| `chemtastes_mol2vec.csv` | Mol2Vec | 300 | Molecular fragment embeddings |
| `chemtastes_rdkit_descriptors.csv` | RDKit 2D | ~200 | Physicochemical descriptors |
| `chemtastes_morgan_fps.csv` | Morgan FP | 2048 | Circular fingerprint bits |
| `chemtastes_maccs.csv` | MACCS Keys | 167 | Structural key fingerprints |
| `chemtastes_chemberta.csv` | ChemBERTa | 768 | Transformer-based embeddings |

## Labels (Multi-label)
- Sweet, Bitter, Umami, Sour, Salty

---
Reference - UmamiPred Data generation file

In [7]:
%pip install rdkit pandas numpy scikit-learn transformers torch tqdm joblib gensim mol2vec

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from mol2vec.features import mol2alt_sentence
from gensim.models import Word2Vec

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./chemtastesdb_multilabel_clean.csv"
MODEL_PATH = "./model_300dim.pkl"
OUTPUT_CSV = "./Embeddings/chemtastes_mol2vec.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Salty']
# ────────────────────────────────────────────────────────────────────────────

def load_chemtastes_csv(path: str) -> pd.DataFrame:
    """Load ChemTastesDB multi-label dataset."""
    df = pd.read_csv(path)
    required_cols = ['canonical SMILES'] + LABEL_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return df

def smiles_to_sentences(smiles_list, radius: int = 1):
    """Convert SMILES to mol2vec fragment sentences."""
    sentences = []
    invalid_count = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            sentences.append([])
            invalid_count += 1
        else:
            sentences.append(mol2alt_sentence(mol, radius=radius))
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    return sentences

def sentences2vec_gensim4(sentences, model: Word2Vec, unseen_vec=None) -> np.ndarray:
    """Custom sentences2vec compatible with Gensim 4.x."""
    vec_dim = model.wv.vector_size
    if unseen_vec is None:
        unseen_vec = np.zeros(vec_dim)
    
    vectors = []
    for sentence in sentences:
        if not sentence:
            vectors.append(unseen_vec)
            continue
        
        word_vecs = []
        for word in sentence:
            if word in model.wv:
                word_vecs.append(model.wv[word])
            else:
                word_vecs.append(unseen_vec)
        
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(unseen_vec)
    
    return np.array(vectors)


if __name__ == "__main__":
    print("=" * 60)
    print("🧬 MOL2VEC EMBEDDING GENERATION")
    print("=" * 60)
    
    print(f"\n[1/5] Loading data from {INPUT_CSV}...")
    df = load_chemtastes_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print(f"\n[2/5] Loading Mol2Vec model from {MODEL_PATH}...")
    model = Word2Vec.load(MODEL_PATH)
    print(f"       Model vector size: {model.wv.vector_size}")

    print("\n[3/5] Converting SMILES to fragment sentences...")
    sentences = smiles_to_sentences(df['canonical SMILES'], radius=1)

    print("\n[4/5] Computing embeddings...")
    vectors = sentences2vec_gensim4(sentences, model)
    n_dims = vectors.shape[1]
    col_names = [f"mol2vec_{i}" for i in range(n_dims)]
    emb_df = pd.DataFrame(vectors, columns=col_names)

    print(f"\n[5/5] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), emb_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done! Output shape: {out_df.shape}")
    print(f"   Columns: {len(LABEL_COLS)} labels + {n_dims} mol2vec features")

🧬 MOL2VEC EMBEDDING GENERATION

[1/5] Loading data from ./chemtastesdb_multilabel_clean.csv...
       Dataset shape: (3849, 7)
       Label distribution:
         • Sweet: 1382 positive (35.9%)
         • Bitter: 1758 positive (45.7%)
         • Umami: 295 positive (7.7%)
         • Sour: 106 positive (2.8%)
         • Salty: 56 positive (1.5%)

[2/5] Loading Mol2Vec model from ./Embeddings/model_300dim.pkl...


c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\gensim\utils.py:1460: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return _pickle.load(f, encoding='latin1')  # needed because loading from S3 doesn't support readline()


       Model vector size: 300

[3/5] Converting SMILES to fragment sentences...


[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerator
[22:04:43] DEPRECATION WARNING: please use MorganGenerat


[4/5] Computing embeddings...

[5/5] Saving to ./Embeddings/chemtastes_mol2vec.csv...

✅ Done! Output shape: (3849, 305)
   Columns: 5 labels + 300 mol2vec features


In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./chemtastesdb_multilabel_clean.csv"
DESCRIPTORS_CSV = "./Embeddings/chemtastes_rdkit_descriptors.csv"
FINGERPRINTS_CSV = "./Embeddings/chemtastes_morgan_fps.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Salty']
# ────────────────────────────────────────────────────────────────────────────

def load_chemtastes_csv(path: str) -> pd.DataFrame:
    """Load ChemTastesDB multi-label dataset."""
    df = pd.read_csv(path)
    required_cols = ['canonical SMILES'] + LABEL_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return df

def compute_rdkit_descriptors(smiles_list) -> pd.DataFrame:
    """Compute all RDKit 2D molecular descriptors."""
    desc_list = Descriptors.descList
    names = [n for n, _ in desc_list]
    
    records = []
    invalid_count = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            records.append([np.nan] * len(desc_list))
            invalid_count += 1
        else:
            records.append([func(mol) for _, func in desc_list])
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    return pd.DataFrame(records, columns=names)

def compute_morgan_fingerprints(smiles_list, radius: int = 2, n_bits: int = 2048) -> pd.DataFrame:
    """Compute Morgan fingerprints as bit vectors."""
    fps = []
    invalid_count = 0
    
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            arr = np.zeros((n_bits,), dtype=int)
            invalid_count += 1
        else:
            bv = rdMolDescriptors.GetMorganFingerprintAsBitVect(
                mol, radius=radius, nBits=n_bits
            )
            arr = np.array(bv, dtype=int)
        fps.append(arr)
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    bitnames = [f"fp_bit_{i}" for i in range(n_bits)]
    return pd.DataFrame(fps, columns=bitnames)


if __name__ == "__main__":
    print("=" * 60)
    print("🧪 RDKIT FEATURES GENERATION")
    print("=" * 60)
    
    print(f"\n[1/4] Loading data from {INPUT_CSV}...")
    df = load_chemtastes_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print("\n[2/4] Computing RDKit 2D descriptors...")
    desc_df = compute_rdkit_descriptors(df['canonical SMILES'])
    out_desc = pd.concat([df[LABEL_COLS].reset_index(drop=True), desc_df], axis=1)
    out_desc.to_csv(DESCRIPTORS_CSV, index=False)
    print(f"       ✓ Wrote {desc_df.shape[1]} descriptors to {DESCRIPTORS_CSV}")

    print("\n[3/4] Computing Morgan fingerprints (radius=2, bits=2048)...")
    fp_df = compute_morgan_fingerprints(df['canonical SMILES'], radius=2, n_bits=2048)
    out_fp = pd.concat([df[LABEL_COLS].reset_index(drop=True), fp_df], axis=1)
    out_fp.to_csv(FINGERPRINTS_CSV, index=False)
    print(f"       ✓ Wrote {fp_df.shape[1]} fingerprint bits to {FINGERPRINTS_CSV}")

    print("\n[4/4] Summary:")
    print(f"       • Descriptors: {out_desc.shape} → {DESCRIPTORS_CSV}")
    print(f"       • Fingerprints: {out_fp.shape} → {FINGERPRINTS_CSV}")
    print("\n✅ Done!")

🧪 RDKIT FEATURES GENERATION

[1/4] Loading data from ./chemtastesdb_multilabel_clean.csv...
       Dataset shape: (3849, 7)
       Label distribution:
         • Sweet: 1382 positive (35.9%)
         • Bitter: 1758 positive (45.7%)
         • Umami: 295 positive (7.7%)
         • Sour: 106 positive (2.8%)
         • Salty: 56 positive (1.5%)

[2/4] Computing RDKit 2D descriptors...
       ✓ Wrote 217 descriptors to ./chemtastes_rdkit_descriptors.csv

[3/4] Computing Morgan fingerprints (radius=2, bits=2048)...


[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerator
[22:05:46] DEPRECATION WARNING: please use MorganGenerat

       ✓ Wrote 2048 fingerprint bits to ./chemtastes_morgan_fps.csv

[4/4] Summary:
       • Descriptors: (3849, 222) → ./chemtastes_rdkit_descriptors.csv
       • Fingerprints: (3849, 2053) → ./chemtastes_morgan_fps.csv

✅ Done!


In [10]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./chemtastesdb_multilabel_clean.csv"
OUTPUT_CSV = "./Embeddings/chemtastes_chemberta.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Salty']
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
BATCH_SIZE = 32
# ────────────────────────────────────────────────────────────────────────────

def load_chemtastes_csv(path: str) -> pd.DataFrame:
    """Load ChemTastesDB multi-label dataset."""
    df = pd.read_csv(path)
    required_cols = ['canonical SMILES'] + LABEL_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return df

def compute_chemberta_embeddings(smiles_list, model_name=MODEL_NAME, 
                                  batch_size=BATCH_SIZE, device=None):
    """
    Compute ChemBERTa embeddings for a list of SMILES strings.
    Uses [CLS] token embedding as the molecule representation.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"       Using device: {device}")
    print(f"       Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    
    all_embeddings = []
    total = len(smiles_list)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            batch_smiles = smiles_list[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(
                batch_smiles.tolist() if hasattr(batch_smiles, 'tolist') else list(batch_smiles),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            # Forward pass
            outputs = model(**inputs)
            
            # Use [CLS] token embedding as molecule representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)
            
            # Progress update
            processed = min(i + batch_size, total)
            if processed % 500 == 0 or processed == total:
                print(f"       Processed {processed}/{total} molecules ({100*processed/total:.1f}%)")
    
    return np.vstack(all_embeddings)


if __name__ == "__main__":
    print("=" * 60)
    print("🤖 CHEMBERTA EMBEDDING GENERATION")
    print("=" * 60)
    
    print(f"\n[1/4] Loading data from {INPUT_CSV}...")
    df = load_chemtastes_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")
    
    print("\n[2/4] Computing ChemBERTa embeddings...")
    embeddings = compute_chemberta_embeddings(df['canonical SMILES'])
    
    n_dims = embeddings.shape[1]
    col_names = [f"chemberta_{i}" for i in range(n_dims)]
    emb_df = pd.DataFrame(embeddings, columns=col_names)
    print(f"       Embedding dimension: {n_dims}")
    
    print(f"\n[3/4] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), emb_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n[4/4] Summary:")
    print(f"       • Output shape: {out_df.shape}")
    print(f"       • Columns: {len(LABEL_COLS)} labels + {n_dims} ChemBERTa features")
    print("\n✅ Done!")

c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 CHEMBERTA EMBEDDING GENERATION

[1/4] Loading data from ./chemtastesdb_multilabel_clean.csv...
       Dataset shape: (3849, 7)
       Label distribution:
         • Sweet: 1382 positive (35.9%)
         • Bitter: 1758 positive (45.7%)
         • Umami: 295 positive (7.7%)
         • Sour: 106 positive (2.8%)
         • Salty: 56 positive (1.5%)

[2/4] Computing ChemBERTa embeddings...
       Using device: cpu
       Loading model: seyonec/ChemBERTa-zinc-base-v1
       Processed 3849/3849 molecules (100.0%)
       Embedding dimension: 768

[3/4] Saving to ./Embeddings/chemtastes_chemberta.csv...

[4/4] Summary:
       • Output shape: (3849, 773)
       • Columns: 5 labels + 768 ChemBERTa features

✅ Done!


In [11]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import MACCSkeys

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./chemtastesdb_multilabel_clean.csv"
OUTPUT_CSV = "./Embeddings/chemtastes_maccs.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Salty']
# ────────────────────────────────────────────────────────────────────────────

def load_chemtastes_csv(path: str) -> pd.DataFrame:
    """Load ChemTastesDB multi-label dataset."""
    df = pd.read_csv(path)
    required_cols = ['canonical SMILES'] + LABEL_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return df

def compute_maccs_fingerprints(smiles_list) -> pd.DataFrame:
    """
    Compute MACCS Keys fingerprints (166 bits).
    
    MACCS keys are structural keys that encode specific substructural patterns:
    - Atom counts and types
    - Ring structures
    - Functional groups
    - Bond patterns
    """
    fps = []
    invalid_count = 0
    
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            # MACCS keys have 167 bits (0-166), but bit 0 is always 0
            arr = np.zeros((167,), dtype=int)
            invalid_count += 1
        else:
            maccs_fp = MACCSkeys.GenMACCSKeys(mol)
            arr = np.array(maccs_fp, dtype=int)
        fps.append(arr)
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    # MACCS keys are 167 bits (indices 0-166)
    bitnames = [f"maccs_{i}" for i in range(167)]
    return pd.DataFrame(fps, columns=bitnames)


if __name__ == "__main__":
    print("=" * 60)
    print("🔑 MACCS KEYS FINGERPRINT GENERATION")
    print("=" * 60)
    
    print(f"\n[1/3] Loading data from {INPUT_CSV}...")
    df = load_chemtastes_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print("\n[2/3] Computing MACCS Keys fingerprints (167 bits)...")
    maccs_df = compute_maccs_fingerprints(df['canonical SMILES'])
    
    # Calculate bit statistics
    bit_density = maccs_df.mean().mean() * 100
    active_bits = (maccs_df.sum(axis=0) > 0).sum()
    print(f"       Average bit density: {bit_density:.2f}%")
    print(f"       Active bits (at least 1 molecule): {active_bits}/167")

    print(f"\n[3/3] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), maccs_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done!")
    print(f"   Output shape: {out_df.shape}")
    print(f"   Columns: {len(LABEL_COLS)} labels + 167 MACCS key bits")

🔑 MACCS KEYS FINGERPRINT GENERATION

[1/3] Loading data from ./chemtastesdb_multilabel_clean.csv...
       Dataset shape: (3849, 7)
       Label distribution:
         • Sweet: 1382 positive (35.9%)
         • Bitter: 1758 positive (45.7%)
         • Umami: 295 positive (7.7%)
         • Sour: 106 positive (2.8%)
         • Salty: 56 positive (1.5%)

[2/3] Computing MACCS Keys fingerprints (167 bits)...
       Average bit density: 24.44%
       Active bits (at least 1 molecule): 159/167

[3/3] Saving to ./Embeddings/chemtastes_maccs.csv...

✅ Done!
   Output shape: (3849, 172)
   Columns: 5 labels + 167 MACCS key bits
